# 01 - Data Preparation

This notebook handles the complete data preparation pipeline:
1. Load raw clinical data
2. Preprocess clinical data
3. Load and preprocess RNA-Seq data
4. Merge datasets by Patient Identifier
5. Run leakage audit
6. Save processed data

> **Note:** All business logic is in `src/`. This notebook only orchestrates.

In [1]:
import sys
from pathlib import Path

# # Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import config
from src.io import load_clinical_raw, save_dataframe, logger
from src.clinical import preprocess_clinical
from src.genomics import preprocess_rna_seq
from src.merge import load_and_merge
from src.leakage import audit_leakage

## Step 1: Load and Preprocess Clinical Data

In [2]:
clinical_raw = load_clinical_raw()
print(f"Raw clinical data shape: {clinical_raw.shape}")
clinical_raw.head(6)

2026-08-14 20:05:30 | INFO     | prostate_bcr | Loading raw clinical data: D:\Prostate_BCR\core\data\raw\data_clinical_patient.tsv
2026-08-14 20:05:30 | INFO     | prostate_bcr | Loaded clinical data: 504 rows × 69 columns


Raw clinical data shape: (504, 69)


,#Other Patient ID,Patient Identifier,Form completion date,Neoplasm Histologic Type Name,Tumor Other Histologic Subtype,Patient Primary Tumor Site,Gleason pattern primary,Gleason pattern secondary,Radical Prostatectomy Gleason Score for Prostate Cancer,Gleason pattern tertiary,...,Race Category,Stage Other,American Joint Committee on Cancer Publication Version Type,Adjuvant Postoperative Targeted Therapy Administered Indicator,Tissue Source Site,Tumor Tissue Site,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
0,#Legacy DMP patient identifier (DMPnnnn),Identifier to uniquely specify a patient.,Form completion date,Text term for the structural pattern of cancer...,Text to describe a tumor's histologic subtype ...,Text term to describe the organ sub-division i...,Gleason pattern primary,Gleason pattern secondary,The score derived from universally embraced pr...,Gleason pattern tertiary,...,The text for reporting information about race.,Stage Other,The version or edition of the American Joint C...,Text term to signify postoperative adjuvant ca...,"A Tissue Source Site collects samples (tissue,...",Text term that describes the anatomic site of ...,Overall patient survival status.,Overall survival in months since initial diago...,Disease free status since initial diagnosis.,Disease free (months) since initial diagnosis.
1,#STRING,STRING,STRING,STRING,STRING,STRING,NUMBER,NUMBER,STRING,NUMBER,...,STRING,STRING,STRING,STRING,STRING,STRING,STRING,NUMBER,STRING,NUMBER
2,#1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
3,OTHER_PATIENT_ID,PATIENT_ID,FORM_COMPLETION_DATE,HISTOLOGICAL_DIAGNOSIS,HISTOLOGICAL_SUBTYPE,PRIMARY_SITE_PATIENT,GLEASON_PATTERN_PRIMARY,GLEASON_PATTERN_SECONDARY,GLEASON_SCORE,GLEASON_PATTERN_TERTIARY,...,RACE,STAGE_OTHER,AJCC_STAGING_EDITION,TARGETED_MOLECULAR_THERAPY,TISSUE_SOURCE_SITE,SITE_OF_TUMOR_TISSUE,OS_STATUS,OS_MONTHS,DFS_STATUS,DFS_MONTHS
4,49197847-CC83-4CE1-8397-D09CEA4C4928,TCGA-2A-A8VL,3/29/14,Prostate Adenocarcinoma Acinar Type,[Not Applicable],Peripheral Zone,3,3,6,4,...,[Not Available],[Not Available],[Not Applicable],NO,2A,Prostate,0:LIVING,20.4,0:DiseaseFree,20.40
5,91C0D161-2B59-4B7A-8C19-6D26DEA31849,TCGA-2A-A8VO,3/30/14,Prostate Adenocarcinoma Acinar Type,[Not Applicable],Overlapping / Multiple Zones,3,3,6,4,...,[Not Available],[Not Available],[Not Applicable],NO,2A,Prostate,0:LIVING,55.88,0:DiseaseFree,55.88


In [3]:
clinical_processed = preprocess_clinical(clinical_raw)
print(f"Processed clinical data shape: {clinical_processed.shape}")
clinical_processed.head()

2026-08-14 20:05:30 | INFO     | prostate_bcr | Dropped 4 metadata rows → 500 patients
2026-08-14 20:05:30 | INFO     | prostate_bcr | Detected 12 numeric / 56 categorical columns
2026-08-14 20:05:30 | INFO     | prostate_bcr | Dropped 21 near-constant columns
2026-08-14 20:05:30 | INFO     | prostate_bcr | Removed 7 leakage columns
2026-08-14 20:05:30 | INFO     | prostate_bcr | Dropped #Other Patient ID (500 levels > 15)
2026-08-14 20:05:30 | INFO     | prostate_bcr | Dropped Form completion date (128 levels > 15)
2026-08-14 20:05:30 | INFO     | prostate_bcr | Dropped Days to bone Scan performed (92 levels > 15)
2026-08-14 20:05:30 | INFO     | prostate_bcr | Dropped Days to ct scan ab pelvis (91 levels > 15)
2026-08-14 20:05:30 | INFO     | prostate_bcr | Dropped Days to mri (68 levels > 15)
2026-08-14 20:05:30 | INFO     | prostate_bcr | Dropped Tissue Source Site (32 levels > 15)
2026-08-14 20:05:30 | INFO     | prostate_bcr | One-hot encoding done → 116 total columns
2026-08-14 

Processed clinical data shape: (431, 116)


,Gleason pattern primary,Gleason pattern secondary,Radical Prostatectomy Gleason Score for Prostate Cancer,Year Cancer Initial Diagnosis,Lymph Node(s) Examined Number,Positive Finding Lymph Node Hematoxylin and Eosin Staining Microscopy Count,Diagnosis Age,Tumor Other Histologic Subtype_25-30% ductal component,Tumor Other Histologic Subtype_Adenocarcinoma prostate with prominent ductal differentiation identified,Tumor Other Histologic Subtype_Mixed,...,American Joint Committee on Cancer Tumor Stage Code_T3a,American Joint Committee on Cancer Tumor Stage Code_T3b,American Joint Committee on Cancer Tumor Stage Code_T4,Race Category_ASIAN,Race Category_BLACK OR AFRICAN AMERICAN,Race Category_WHITE,Adjuvant Postoperative Targeted Therapy Administered Indicator_NO,Adjuvant Postoperative Targeted Therapy Administered Indicator_YES,Patient Identifier,Biochemical_Recurrence_Code
0,3,3,6,2010.0,15.0,0.0,51,0,0,0,...,0,0,0,0,0,0,1,0,TCGA-2A-A8VL,0
1,3,3,6,2010.0,9.0,0.0,57,0,0,0,...,1,0,0,0,0,0,1,0,TCGA-2A-A8VO,0
2,4,5,9,2011.0,16.0,2.0,47,0,0,0,...,0,0,1,0,0,0,0,1,TCGA-2A-A8VT,0
3,3,3,6,2010.0,16.0,0.0,52,0,0,0,...,0,0,0,0,0,0,1,0,TCGA-2A-A8VV,0
4,4,4,8,2011.0,16.0,0.0,70,0,0,0,...,0,1,0,0,0,0,1,0,TCGA-2A-A8VX,0


In [4]:
save_dataframe(clinical_processed, config.CLINICAL_PROCESSED.name)
print("Clinical data saved successfully.")

2026-08-14 20:05:30 | INFO     | prostate_bcr | Saved 431 rows → D:\Prostate_BCR\core\data\processed\clinical_data_complete.csv


Clinical data saved successfully.


## Step 2: Load and Preprocess RNA-Seq Data

In [5]:
rna_seq_processed = preprocess_rna_seq()
print(f"Processed RNA-Seq data shape: {rna_seq_processed.shape}")
rna_seq_processed.head()

2026-08-14 20:05:30 | INFO     | prostate_bcr | Loading RNA-Seq data from D:\Prostate_BCR\core\data\raw\data_mrna_seq_v2_rsem.txt
2026-08-14 20:05:31 | INFO     | prostate_bcr | Dropped metadata columns: ['Entrez_Gene_Id']
2026-08-14 20:05:31 | INFO     | prostate_bcr | Loaded RNA-Seq matrix: 20531 genes × 498 samples
2026-08-14 20:05:31 | INFO     | prostate_bcr | Found 17 duplicate gene names
2026-08-14 20:05:31 | INFO     | prostate_bcr | Resolved duplicates → 20514 unique genes
2026-08-14 20:05:31 | INFO     | prostate_bcr | Filtered low-expression genes: 20514 → 18905 (removed 1609)
2026-08-14 20:05:31 | INFO     | prostate_bcr | Removed 1 duplicate patients from index
2026-08-14 20:05:31 | INFO     | prostate_bcr | Transposed and standardized: 497 samples × 18905 genes
2026-08-14 20:05:31 | INFO     | prostate_bcr | QC Report:
2026-08-14 20:05:31 | INFO     | prostate_bcr |   n_samples: 497.00
2026-08-14 20:05:32 | INFO     | prostate_bcr |   n_genes: 18905.00
2026-08-14 20:05:32

Processed RNA-Seq data shape: (497, 18905)


Hugo_Symbol,KLK3,KLK2,ACPP,EEF1A1,EEF2,RPL3,TPT1,ACTB,MYH11,DFNA26,...,GJA10,LINC00298,SNORA84,RETNLB,PRR21,DMBT1L1,SNAR-B2,BIRC8,TRIM43,TGIF2LY
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-2A-A8VL,586951.5963,211670.4428,322984.0371,94756.9516,73802.2657,60763.1308,102248.7127,63055.6128,78450.2575,29120.4943,...,0.0,0.0000,0.0000,0.0,0.0,1.0299,0.0000,0.0,0.0,0.0
TCGA-2A-A8VO,628137.4305,266434.2706,58250.1171,128735.2579,98047.6733,80067.6585,93103.5514,65646.6937,52082.3267,67617.2974,...,0.0,0.0000,0.6646,0.0,0.0,0.0000,0.6646,0.0,0.0,0.0
TCGA-2A-A8VT,119260.6618,76452.2059,31068.7500,47778.6765,68504.7794,25816.9118,43450.7353,33234.5588,12044.9007,44587.8676,...,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.3676,0.0,0.0,0.0
TCGA-2A-A8VV,603737.2350,195684.6688,186556.5361,154442.6403,131972.2320,98692.0814,153221.7908,66259.0893,105385.5230,49581.3625,...,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0
TCGA-2A-A8VX,360174.8482,173900.4912,137600.2015,126811.9411,126027.4594,112007.0538,94858.2945,59875.0472,45340.3703,64025.6959,...,0.0,1.0077,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0


## Step 3: Merge Datasets by Patient Identifier

In [6]:
X_merged, y = load_and_merge()
print(f"Merged data shape: {X_merged.shape}")
print(f"Target distribution:\n{y.value_counts().sort_index()}")
print(f"Positive rate: {y.mean():.2%}")

2026-08-14 20:05:32 | INFO     | prostate_bcr | Loading processed clinical data: D:\Prostate_BCR\core\data\processed\clinical_data_complete.csv
2026-08-14 20:05:32 | INFO     | prostate_bcr | Loading RNA-Seq data from D:\Prostate_BCR\core\data\raw\data_mrna_seq_v2_rsem.txt
2026-08-14 20:05:33 | INFO     | prostate_bcr | Dropped metadata columns: ['Entrez_Gene_Id']
2026-08-14 20:05:33 | INFO     | prostate_bcr | Loaded RNA-Seq matrix: 20531 genes × 498 samples
2026-08-14 20:05:33 | INFO     | prostate_bcr | Found 17 duplicate gene names
2026-08-14 20:05:33 | INFO     | prostate_bcr | Resolved duplicates → 20514 unique genes
2026-08-14 20:05:33 | INFO     | prostate_bcr | Filtered low-expression genes: 20514 → 18905 (removed 1609)
2026-08-14 20:05:33 | INFO     | prostate_bcr | Removed 1 duplicate patients from index
2026-08-14 20:05:33 | INFO     | prostate_bcr | Transposed and standardized: 497 samples × 18905 genes
2026-08-14 20:05:33 | INFO     | prostate_bcr | QC Report:
2026-08-14 

Merged data shape: (429, 19019)
Target distribution:
Biochemical_Recurrence_Code
0    371
1     58
Name: count, dtype: int64
Positive rate: 13.52%


## Step 4: Leakage Audit

In [7]:
X_merged_with_target = X_merged.copy()
X_merged_with_target[config.TARGET_COLUMN] = y

X_clean, leakage_report = audit_leakage(
    X_merged_with_target,
    drop_columns=True,
    raw_clinical_path=config.CLINICAL_RAW,
    output_path=config.TABLES_DIR / "leakage_report.csv",
)

print(f"\nData shape after leakage audit: {X_clean.shape}")

2026-08-14 20:05:33 | INFO     | prostate_bcr | ============================================================
2026-08-14 20:05:33 | INFO     | prostate_bcr | LEAKAGE AUDIT REPORT
2026-08-14 20:05:33 | INFO     | prostate_bcr | ============================================================
2026-08-14 20:05:33 | INFO     | prostate_bcr | No known leakage columns detected
2026-08-14 20:05:33 | WARNING  | prostate_bcr | Column 'Patient Identifier' not found; skipping duplicate check
2026-08-14 20:05:33 | INFO     | prostate_bcr | Running temporal PSA audit on D:\Prostate_BCR\core\data\raw\data_clinical_patient.tsv
2026-08-14 20:05:33 | WARNING  | prostate_bcr | YES cases with PSA at/after recurrence: 52/52
2026-08-14 20:05:33 | INFO     | prostate_bcr | ------------------------------------------------------------
2026-08-14 20:05:33 | INFO     | prostate_bcr | Summary:
2026-08-14 20:05:33 | INFO     | prostate_bcr |   Samples: 429
2026-08-14 20:05:33 | INFO     | prostate_bcr |   Features: 19


Data shape after leakage audit: (429, 19020)


## Step 5: Save Final Dataset

In [8]:
X_final = X_clean.drop(columns=[config.TARGET_COLUMN], errors="ignore")
y_final = X_clean[config.TARGET_COLUMN]

save_dataframe(X_final, "X_features_final.csv", index=False)
save_dataframe(y_final.to_frame(), "y_target_final.csv", index=False)

print(f"Final features shape: {X_final.shape}")
print(f"Final target shape: {y_final.shape}")
print(f"Positive rate: {y_final.mean():.2%}")

2026-08-14 20:05:40 | INFO     | prostate_bcr | Saved 429 rows → D:\Prostate_BCR\core\data\processed\X_features_final.csv
2026-08-14 20:05:40 | INFO     | prostate_bcr | Saved 429 rows → D:\Prostate_BCR\core\data\processed\y_target_final.csv


Final features shape: (429, 19019)
Final target shape: (429,)
Positive rate: 13.52%
